# GraphRAG Retrieval Comparison — Scoring & Analysis

This notebook reads `data/eval/results.json` (produced by `evaluate.py`) and scores each retrieval method.

**Methods compared:** graph | vector | hybrid  
**Metrics:** hit count, correctness, latency

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Load results and query metadata

In [ ]:
results_path = Path("../data/eval/results.json")
queries_path = Path("../data/eval/queries.json")

with results_path.open() as f:
    results = json.load(f)

with queries_path.open() as f:
    queries = json.load(f)

# Build a lookup so we can join query type and expected answer into results
query_meta = {q["question"]: q for q in queries}

print(f"Loaded {len(results)} result(s) and {len(queries)} query definition(s)")

## 2. Build a flat DataFrame

One row per (question, method) — easier to slice and plot.

In [ ]:
rows = []
for r in results:
    q = r["question"]
    meta = query_meta.get(q, {})
    expected = (r.get("expected") or meta.get("expected") or "").lower()
    q_type = meta.get("type", "unknown")
    q_id = meta.get("id", "?")

    graph = r["graph"]
    vector = r["vector"]
    hybrid = r["hybrid"]

    for method, hit_count, latency in [
        ("graph",  graph["hit_count"],  graph["latency_sec"]),
        ("vector", vector["hit_count"], vector["latency_sec"]),
        ("hybrid", hybrid["graph_hit_count"] + hybrid["doc_hit_count"], hybrid["latency_sec"]),
    ]:
        rows.append({
            "id": q_id,
            "question": q,
            "type": q_type,
            "depth": r.get("depth", 1),
            "expected": expected,
            "method": method,
            "hit_count": hit_count,
            "latency_sec": latency,
        })

df = pd.DataFrame(rows)
df.head(9)

## 3. Correctness scoring

We check whether the expected answer string appears anywhere in the retrieved context text.

> **Note:** `evaluate.py` currently stores hit *counts*, not the actual text of hits.
> Once Member 4 wires the LLM answer into results, swap `hits_contain_expected()` to check
> the generated answer string instead. For now we use hit_count > 0 as a proxy.

In [ ]:
def hits_contain_expected(row):
    """Proxy: at least one hit was returned. Replace with answer-text check once LLM is wired."""
    return int(row["hit_count"] > 0)

df["correct"] = df.apply(hits_contain_expected, axis=1)
df[["id", "question", "method", "hit_count", "correct"]].sort_values(["id", "method"])

## 4. Summary table — accuracy and latency per method

In [ ]:
summary = (
    df.groupby("method")
    .agg(
        accuracy=("correct", "mean"),
        avg_hits=("hit_count", "mean"),
        avg_latency_ms=("latency_sec", lambda x: x.mean() * 1000),
    )
    .round(3)
    .sort_values("accuracy", ascending=False)
)
summary

## 5. Breakdown by query type (simple vs multi-hop vs vector-favored)

In [ ]:
type_summary = (
    df.groupby(["type", "method"])
    .agg(accuracy=("correct", "mean"), avg_latency_ms=("latency_sec", lambda x: x.mean() * 1000))
    .round(3)
    .reset_index()
)
type_summary.pivot(index="type", columns="method", values="accuracy")

## 6. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy bar chart
summary["accuracy"].plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Accuracy by Method")
axes[0].set_ylabel("Accuracy (fraction of queries with hits)")
axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Latency bar chart
summary["avg_latency_ms"].plot(kind="bar", ax=axes[1], color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title("Avg Latency by Method (ms)")
axes[1].set_ylabel("Milliseconds")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig("../reports/retrieval_comparison.png", dpi=150)
plt.show()
print("Chart saved to reports/retrieval_comparison.png")

In [ ]:
# Per-question view: who got it right?
pivot = df.pivot_table(index=["id", "question", "type"], columns="method", values="correct")
pivot.columns.name = None
pivot = pivot[["graph", "vector", "hybrid"]]
pivot["any_miss"] = (pivot < 1).any(axis=1)  # flag questions where at least one method failed
pivot

## 7. Key observations — fill this in after running

Replace the placeholder bullets below with your actual findings.

- **Graph** performed best on: *(e.g., multi-hop questions requiring 2+ traversal steps)*
- **Vector** performed best on: *(e.g., questions whose answers live in document text, not graph edges)*
- **Hybrid** trade-off: *(e.g., matched or beat both on most queries but added X ms latency)*
- Biggest gap between methods: *(e.g., q06 — 2-hop accessory question — graph 1, vector 0)*
- Latency winner: *(e.g., graph was fastest at X ms avg)*